In [28]:
import torch 
import torchvision.models as models
from sentiment import BertClassifier

In [29]:
model = torch.load('bert_multilabel_best_model.pt',map_location=torch.device('cpu'))

In [30]:
sentence = "Magnifique épopée, une belle histoire, touchante avec des acteurs qui interprètent très bien leur rôles (Mel Gibson, Heath Ledger, Jason Isaacs...), le genre de film qui se savoure en famille! :)"

In [31]:
type(model)

collections.OrderedDict

In [32]:
def load_bert_classifier_from_checkpoint(checkpoint_path, model_name="bert-base-uncased", n_classes=2, device="cpu"):
    """Load a saved state_dict or checkpoint into the BertClassifier architecture."""
    model = BertClassifier(model_name=model_name, n_classes=n_classes)
    checkpoint = torch.load(checkpoint_path, map_location=device)

    if isinstance(checkpoint, dict):
        if "state_dict" in checkpoint:
            state_dict = checkpoint["state_dict"]
        elif "model_state_dict" in checkpoint:
            state_dict = checkpoint["model_state_dict"]
        elif "model" in checkpoint and isinstance(checkpoint["model"], dict):
            state_dict = checkpoint["model"]
        else:
            state_dict = checkpoint
    else:
        return checkpoint.to(device).eval()

    cleaned_state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}
    missing, unexpected = model.load_state_dict(cleaned_state_dict, strict=False)
    if missing:
        print(f"Missing keys while loading: {missing}")
    if unexpected:
        print(f"Unexpected keys while loading: {unexpected}")

    model.to(device)
    model.eval()
    return model


In [34]:
checkpoint_path = "bert_multilabel_best_model.pt"
model = load_bert_classifier_from_checkpoint(
    checkpoint_path,
    model_name="bert-base-uncased",
    n_classes=2,
    device="cpu",
)

print(type(model))
print(model)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8680.88it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


RuntimeError: Error(s) in loading state_dict for BertClassifier:
	size mismatch for classifier.weight: copying a param with shape torch.Size([3, 768]) from checkpoint, the shape in current model is torch.Size([2, 768]).
	size mismatch for classifier.bias: copying a param with shape torch.Size([3]) from checkpoint, the shape in current model is torch.Size([2]).